# Classification Fundamentals: k-NN, Naive Bayes, and Decision Trees

Four small classification studies implemented with Python and Pandas: k-Nearest Neighbors with Manhattan distance, categorical Naive Bayes, mixed-feature Naive Bayes with a Gaussian likelihood, and a Decision Tree split analysis using entropy and information gain.

The examples use small, manually defined datasets to make the calculations and predictions transparent. They are educational implementations rather than trained and validated production classifiers.

## 1. Libraries

Standard Python utilities and Pandas support distance calculations, probability estimates, tables, and entropy calculations.

In [ ]:
import math
import pandas as pd
from collections import Counter
from pprint import pprint

## 2. T-Shirt Size Classification with k-NN

This example predicts a T-shirt size (**M** or **L**) using two features: height in centimeters and weight in kilograms. Manhattan distance ranks the training observations by their proximity to a sample of **(161 cm, 63 kg)**.

**Manhattan distance**

$$
D(x,y) = |x_1-y_1| + |x_2-y_2|
$$

The classifier selects the most common label among the **k** nearest records. Both **k = 3** and **k = 7** are examined. Height and weight are used directly without feature scaling.

In [ ]:
# Data training soal 1
data_knn = [
    [158, 58, 'M'],
    [158, 59, 'M'],
    [158, 60, 'M'],
    [160, 58, 'M'],
    [160, 60, 'M'],
    [163, 60, 'M'],
    [163, 61, 'M'],
    [160, 64, 'L'],
    [163, 64, 'L'],
    [165, 61, 'L'],
    [165, 62, 'L'],
    [164, 63, 'L'],
    [168, 62, 'L'],
    [168, 63, 'L'],
    [168, 66, 'L'],
    [170, 63, 'L'],
    [170, 64, 'L'],
    [170, 68, 'L'],
]

df_knn = pd.DataFrame(data_knn, columns=['Tinggi', 'Berat', 'Ukuran'])

# Data uji
test_height = 161
test_weight = 63

# Hitung Manhattan Distance
df_knn['Jarak_Manhattan'] = (df_knn['Tinggi'] - test_height).abs() + (df_knn['Berat'] - test_weight).abs()

# Urutkan berdasarkan jarak terkecil
df_knn_sorted = df_knn.sort_values(by='Jarak_Manhattan').reset_index(drop=True)

print("Data dengan Manhattan Distance ke titik uji (161, 63):")
display(df_knn_sorted)

Data dengan Manhattan Distance ke titik uji (161, 63):


,Tinggi,Berat,Ukuran,Jarak_Manhattan
0,160,64,L,2
1,163,64,L,3
2,164,63,L,3
3,160,60,M,4
4,163,61,M,4
5,165,62,L,5
6,163,60,M,5
7,158,60,M,6
8,160,58,M,6
9,165,61,L,6


In [ ]:
def predict_knn(df_sorted, k):
    neighbors = df_sorted.head(k).copy()
    vote_count = neighbors['Ukuran'].value_counts()
    prediction = vote_count.idxmax()
    return neighbors, vote_count, prediction

# Prediksi untuk K=3
neighbors_k3, vote_k3, pred_k3 = predict_knn(df_knn_sorted, 3)

# Prediksi untuk K=7
neighbors_k7, vote_k7, pred_k7 = predict_knn(df_knn_sorted, 7)

print("=== Tetangga terdekat untuk K=3 ===")
display(neighbors_k3)
print("Voting K=3:")
print(vote_k3)
print("Prediksi K=3:", pred_k3)

print("\n=== Tetangga terdekat untuk K=7 ===")
display(neighbors_k7)
print("Voting K=7:")
print(vote_k7)
print("Prediksi K=7:", pred_k7)

=== Tetangga terdekat untuk K=3 ===


,Tinggi,Berat,Ukuran,Jarak_Manhattan
0,160,64,L,2
1,163,64,L,3
2,164,63,L,3


Voting K=3:
Ukuran
L    3
Name: count, dtype: int64
Prediksi K=3: L

=== Tetangga terdekat untuk K=7 ===


,Tinggi,Berat,Ukuran,Jarak_Manhattan
0,160,64,L,2
1,163,64,L,3
2,164,63,L,3
3,160,60,M,4
4,163,61,M,4
5,165,62,L,5
6,163,60,M,5


Voting K=7:
Ukuran
L    4
M    3
Name: count, dtype: int64
Prediksi K=7: L


### k-NN Results

| Parameter | Predicted Size | Nearest-Neighbor Votes |
|---|---|---|
| k = 3 | L | 3 L |
| k = 7 | L | 4 L, 3 M |

Both neighborhood sizes predict **L** for the sample (161, 63). These are predictions for a single sample; the notebook does not measure generalization accuracy.

## 3. Categorical Naive Bayes for Outcome Prediction

This example predicts a **Sick** or **Fine** outcome using three binary categorical inputs: Apple, Beans, and Carrots. Class priors and feature likelihoods are computed from a small table of observations.

**Naive Bayes decision rule**

$$
P(c\mid x) \propto P(c)\prod_i P(x_i\mid c)
$$

The implementation assumes conditional independence between the three features and does **not** apply Laplace smoothing.

In [ ]:
data_nb_cat = [
    ['No',  'Yes', 'No',  'Sick'],
    ['No',  'Yes', 'Yes', 'Sick'],
    ['No',  'Yes', 'No',  'Sick'],
    ['Yes', 'Yes', 'Yes', 'Sick'],
    ['Yes', 'Yes', 'No',  'Fine'],
    ['No',  'No',  'Yes', 'Fine'],
    ['Yes', 'No',  'No',  'Fine'],
    ['No',  'No',  'Yes', 'Fine'],
]

df_nb_cat = pd.DataFrame(data_nb_cat, columns=['Apple', 'Beans', 'Carrots', 'Outcome'])
display(df_nb_cat)

,Apple,Beans,Carrots,Outcome
0,No,Yes,No,Sick
1,No,Yes,Yes,Sick
2,No,Yes,No,Sick
3,Yes,Yes,Yes,Sick
4,Yes,Yes,No,Fine
5,No,No,Yes,Fine
6,Yes,No,No,Fine
7,No,No,Yes,Fine


In [ ]:
# Prior probability
class_counts = df_nb_cat['Outcome'].value_counts()
total_data = len(df_nb_cat)

priors = {cls: count / total_data for cls, count in class_counts.items()}

print("Prior probability:")
for cls, prob in priors.items():
    print(f"P({cls}) = {prob:.4f}")

# Conditional probability tanpa smoothing
features = ['Apple', 'Beans', 'Carrots']
classes = df_nb_cat['Outcome'].unique()

conditional_probs = {}

for cls in classes:
    conditional_probs[cls] = {}
    df_cls = df_nb_cat[df_nb_cat['Outcome'] == cls]

    for feature in features:
        conditional_probs[cls][feature] = {}
        for value in df_nb_cat[feature].unique():
            prob = (df_cls[feature] == value).sum() / len(df_cls)
            conditional_probs[cls][feature][value] = prob

print("\nConditional probability:")
for cls in conditional_probs:
    print(f"\nKelas: {cls}")
    for feature in conditional_probs[cls]:
        for value, prob in conditional_probs[cls][feature].items():
            print(f"P({feature}={value} | {cls}) = {prob:.4f}")

Prior probability:
P(Sick) = 0.5000
P(Fine) = 0.5000

Conditional probability:

Kelas: Sick
P(Apple=No | Sick) = 0.7500
P(Apple=Yes | Sick) = 0.2500
P(Beans=Yes | Sick) = 1.0000
P(Beans=No | Sick) = 0.0000
P(Carrots=No | Sick) = 0.5000
P(Carrots=Yes | Sick) = 0.5000

Kelas: Fine
P(Apple=No | Fine) = 0.5000
P(Apple=Yes | Fine) = 0.5000
P(Beans=Yes | Fine) = 0.2500
P(Beans=No | Fine) = 0.7500
P(Carrots=No | Fine) = 0.5000
P(Carrots=Yes | Fine) = 0.5000


In [ ]:
def predict_naive_bayes_categorical(sample, priors, conditional_probs):
    scores = {}
    for cls in priors:
        score = priors[cls]
        for feature, value in sample.items():
            score *= conditional_probs[cls][feature].get(value, 0)
        scores[cls] = score
    prediction = max(scores, key=scores.get)
    return scores, prediction

sample_1 = {'Apple': 'Yes', 'Beans': 'No', 'Carrots': 'Yes'}
sample_2 = {'Apple': 'No', 'Beans': 'No', 'Carrots': 'No'}

scores_1, pred_1 = predict_naive_bayes_categorical(sample_1, priors, conditional_probs)
scores_2, pred_2 = predict_naive_bayes_categorical(sample_2, priors, conditional_probs)

print("Sample 1:", sample_1)
print("Scores:", scores_1)
print("Prediksi:", pred_1)

print("\nSample 2:", sample_2)
print("Scores:", scores_2)
print("Prediksi:", pred_2)

Sample 1: {'Apple': 'Yes', 'Beans': 'No', 'Carrots': 'Yes'}
Scores: {'Sick': np.float64(0.0), 'Fine': np.float64(0.09375)}
Prediksi: Fine

Sample 2: {'Apple': 'No', 'Beans': 'No', 'Carrots': 'No'}
Scores: {'Sick': np.float64(0.0), 'Fine': np.float64(0.09375)}
Prediksi: Fine


### Categorical Naive Bayes Results

In this dataset, every **Sick** observation has `Beans = Yes`, giving $P(Beans=Yes\mid Sick)=1$ and $P(Beans=No\mid Sick)=0$. This makes Beans highly discriminative **within the sample**, but it does not establish a real-world causal relationship.

| Apple | Beans | Carrots | Predicted Outcome |
|---|---|---|---|
| Yes | No | Yes | Fine |
| No | No | No | Fine |

The absence of smoothing can force a class score to zero when a feature value has not appeared in that class.

## 4. Naive Bayes with Categorical and Numeric Features

This example combines a categorical feature `X1` with a numeric feature `X2` to predict either **+** or **−**. The numeric likelihood is modeled with a Gaussian distribution; the categorical likelihood is estimated from observed frequencies.

**Gaussian likelihood**

$$
P(x\mid c)=\frac{1}{\sqrt{2\pi}\sigma_c}\exp\left(-\frac{(x-\mu_c)^2}{2\sigma_c^2}\right)
$$

The implementation uses these specified parameters: $\mu_+=1.73$, $\mu_-=4.45$, $\sigma_+=1.10$, and $\sigma_-=0.07$. The dataset is small and no smoothing is used.

In [ ]:
data_nb_mix = [
    ['A', 1.0, '+'],
    ['B', 1.2, '+'],
    ['A', 3.0, '+'],
    ['B', 4.4, '-'],
    ['B', 4.5, '-'],
]

df_nb_mix = pd.DataFrame(data_nb_mix, columns=['X1', 'X2', 'Kelas'])
display(df_nb_mix)

,X1,X2,Kelas
0,A,1.0,+
1,B,1.2,+
2,A,3.0,+
3,B,4.4,-
4,B,4.5,-


In [ ]:
# Prior kelas
class_counts_mix = df_nb_mix['Kelas'].value_counts()
total_mix = len(df_nb_mix)

priors_mix = {cls: count / total_mix for cls, count in class_counts_mix.items()}

print("Prior kelas:")
for cls, prob in priors_mix.items():
    print(f"P({cls}) = {prob:.4f}")

# Conditional probability untuk X1
conditional_x1 = {}
for cls in df_nb_mix['Kelas'].unique():
    conditional_x1[cls] = {}
    df_cls = df_nb_mix[df_nb_mix['Kelas'] == cls]
    for value in df_nb_mix['X1'].unique():
        conditional_x1[cls][value] = (df_cls['X1'] == value).sum() / len(df_cls)

print("\nConditional probability untuk X1:")
for cls in conditional_x1:
    for value in conditional_x1[cls]:
        print(f"P(X1={value} | {cls}) = {conditional_x1[cls][value]:.4f}")

Prior kelas:
P(+) = 0.6000
P(-) = 0.4000

Conditional probability untuk X1:
P(X1=A | +) = 0.6667
P(X1=B | +) = 0.3333
P(X1=A | -) = 0.0000
P(X1=B | -) = 1.0000


In [ ]:
# Parameter Gaussian dari soal
mu = {'+': 1.73, '-': 4.45}
sigma = {'+': 1.10, '-': 0.07}

def gaussian_pdf(x, mean, std):
    coef = 1 / (math.sqrt(2 * math.pi) * std)
    exponent = math.exp(-((x - mean) ** 2) / (2 * std ** 2))
    return coef * exponent

def predict_nb_mixed(x1_value, x2_value):
    scores = {}
    for cls in priors_mix:
        score = priors_mix[cls]
        score *= conditional_x1[cls].get(x1_value, 0)
        score *= gaussian_pdf(x2_value, mu[cls], sigma[cls])
        scores[cls] = score
    prediction = max(scores, key=scores.get)
    return scores, prediction

test_1_scores, test_1_pred = predict_nb_mixed('B', 2.0)
test_2_scores, test_2_pred = predict_nb_mixed('A', 4.0)

print("Data uji 1: X1=B, X2=2.0")
print("Scores:", test_1_scores)
print("Prediksi:", test_1_pred)

print("\nData uji 2: X1=A, X2=4.0")
print("Scores:", test_2_scores)
print("Prediksi:", test_2_pred)

Data uji 1: X1=B, X2=2.0
Scores: {'+': np.float64(0.07038250238451439), '-': np.float64(2.2516550155062997e-266)}
Prediksi: +

Data uji 2: X1=A, X2=4.0
Scores: {'+': np.float64(0.017251831529042878), '-': np.float64(0.0)}
Prediksi: +


### Mixed-Feature Naive Bayes Results

| X1 | X2 | Predicted Class |
|---|---:|---|
| B | 2.0 | + |
| A | 4.0 | + |

The negative class has no training record with `X1 = A`, so its unsmoothed categorical likelihood is zero for that input. These outcomes illustrate the method's sensitivity to sparse data and zero probabilities.

## 5. Decision Tree Analysis for Hiking Decisions

The final example models a binary **GoHiking (Y/N)** outcome using Weekend, Company, and Weather. It calculates entropy and information gain to identify a useful first split.

**Entropy**

$$
H(S)=-\sum_c p(c)\log_2 p(c)
$$

**Information gain**

$$
IG(S,A)=H(S)-\sum_v\frac{|S_v|}{|S|}H(S_v)
$$

Higher information gain indicates a larger reduction in uncertainty on the given dataset.

In [ ]:
data_dt = [
    ['Y', 'N', 'R', 'N'],
    ['Y', 'Y', 'R', 'N'],
    ['Y', 'Y', 'R', 'Y'],
    ['Y', 'Y', 'S', 'Y'],
    ['Y', 'N', 'S', 'Y'],
    ['Y', 'N', 'S', 'N'],
    ['Y', 'Y', 'R', 'N'],
    ['Y', 'Y', 'S', 'Y'],
    ['N', 'Y', 'S', 'N'],
    ['N', 'Y', 'R', 'N'],
    ['N', 'N', 'S', 'N'],
]

df_dt = pd.DataFrame(data_dt, columns=['Weekend', 'Company', 'Weather', 'GoHiking'])
display(df_dt)

,Weekend,Company,Weather,GoHiking
0,Y,N,R,N
1,Y,Y,R,N
2,Y,Y,R,Y
3,Y,Y,S,Y
4,Y,N,S,Y
5,Y,N,S,N
6,Y,Y,R,N
7,Y,Y,S,Y
8,N,Y,S,N
9,N,Y,R,N


In [ ]:
def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    ent = 0
    for count in counts.values():
        p = count / total
        if p > 0:
            ent -= p * math.log2(p)
    return ent

def information_gain(df, feature, target='GoHiking'):
    total_entropy = entropy(df[target])
    values = df[feature].unique()

    weighted_entropy = 0
    for value in values:
        subset = df[df[feature] == value]
        weighted_entropy += (len(subset) / len(df)) * entropy(subset[target])

    gain = total_entropy - weighted_entropy
    return gain

print("Entropy total:", round(entropy(df_dt['GoHiking']), 4))

for feature in ['Weekend', 'Company', 'Weather']:
    gain = information_gain(df_dt, feature)
    print(f"Information Gain untuk {feature}: {gain:.4f}")

Entropy total: 0.9457
Information Gain untuk Weekend: 0.2184
Information Gain untuk Company: 0.0237
Information Gain untuk Weather: 0.0721


### Root Feature Selection

| Feature | Information Gain |
|---|---:|
| Weekend | 0.2184 |
| Weather | 0.0721 |
| Company | 0.0237 |

**Weekend** has the highest information gain and is selected as the root feature.

In [ ]:
# Melihat distribusi berdasarkan atribut
for feature in ['Weekend', 'Company', 'Weather']:
    print(f"\n=== Crosstab untuk {feature} ===")
    display(pd.crosstab(df_dt[feature], df_dt['GoHiking']))


=== Crosstab untuk Weekend ===


GoHiking,N,Y
Weekend,,
N,3,0
Y,4,4



=== Crosstab untuk Company ===


GoHiking,N,Y
Company,,
N,3,1
Y,4,3



=== Crosstab untuk Weather ===


GoHiking,N,Y
Weather,,
R,4,1
S,3,3


### Feature Distributions

Cross-tabulations show the class counts for each feature value. They help explain why the Weekend split gives the largest reduction in entropy.

In [ ]:
# Bangun tree sederhana secara manual dari hasil analisis data

# Subset Weather = R
subset_R = df_dt[df_dt['Weather'] == 'R']
print("Subset Weather = R")
display(subset_R)
print("IG di subset Weather=R")
for feature in ['Weekend', 'Company']:
    print(feature, round(information_gain(subset_R, feature), 4))

# Subset Weather = S
subset_S = df_dt[df_dt['Weather'] == 'S']
print("\nSubset Weather = S")
display(subset_S)
print("IG di subset Weather=S")
for feature in ['Weekend', 'Company']:
    print(feature, round(information_gain(subset_S, feature), 4))

Subset Weather = R


,Weekend,Company,Weather,GoHiking
0,Y,N,R,N
1,Y,Y,R,N
2,Y,Y,R,Y
6,Y,Y,R,N
9,N,Y,R,N


IG di subset Weather=R
Weekend 0.0729
Company 0.0729

Subset Weather = S


,Weekend,Company,Weather,GoHiking
3,Y,Y,S,Y
4,Y,N,S,Y
5,Y,N,S,N
7,Y,Y,S,Y
8,N,Y,S,N
10,N,N,S,N


IG di subset Weather=S
Weekend 0.4591
Company 0.0817


### Decision Rules

All observed records with `Weekend = N` have `GoHiking = N`, so this branch directly predicts **No**. The `Weekend = Y` branch contains mixed outcomes and would require additional splits for a complete decision tree.

The current code explores selected subsets but does not implement an automatic full-tree induction algorithm.

In [ ]:
# Prediksi data uji: [Weekend = N, Company = N, Weather = R]
test_case = {'Weekend': 'N', 'Company': 'N', 'Weather': 'R'}

# Cari data training yang paling relevan untuk cabang Weather=R dan Company=N
relevant_subset = df_dt[(df_dt['Weather'] == 'R') & (df_dt['Company'] == 'N')]

print("Data paling relevan untuk test case:")
display(relevant_subset)

if len(relevant_subset) > 0:
    final_prediction = relevant_subset['GoHiking'].mode()[0]
else:
    # fallback: gunakan mayoritas pada Weather = R
    final_prediction = df_dt[df_dt['Weather'] == 'R']['GoHiking'].mode()[0]

print("Prediksi untuk [Weekend=N, Company=N, Weather=R] adalah:", final_prediction)

Data paling relevan untuk test case:


,Weekend,Company,Weather,GoHiking
0,Y,N,R,N


Prediksi untuk [Weekend=N, Company=N, Weather=R] adalah: N


### Hiking Prediction

| Weekend | Company | Weather | Predicted GoHiking |
|---|---|---|---|
| N | N | R | No |

The result follows the `Weekend = N` branch. The notebook demonstrates split selection and one prediction; it does not report accuracy on an independent test set.

## 6. Summary

The four studies cover distance-based classification, categorical probabilities, continuous-feature likelihoods, and information-gain-based decision rules. Each example exposes intermediate calculations, making the methods easier to inspect. Because the datasets are small and no holdout evaluation is performed, these results are demonstrations rather than performance benchmarks.